In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

In [33]:
# Selenium 웹드라이버 설정
options = webdriver.ChromeOptions()
options.add_argument('--disable-gpu')
#options.add_argument('--blink-settings=imagesEnabled=false') # 이미지 로딩 차단 
#options.add_argument('--headless')  # 헤드리스 모드 즉 백그라운드 실행 (선택사항)
#driver = webdriver.Chrome(options=options)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [34]:
# 2. 페이지 접속 및 리뷰 팝업 열기

url = "https://play.google.com/store/apps/details?id=com.nianticlabs.pokemongo"
driver.get(url)
time.sleep(1) 

In [35]:
# '리뷰 모두 보기' 버튼 클릭 (사용자님의 셀렉터 활용)
#btn_selector = '#yDmH0d > c-wiz.SSPGKf.Czez9d > div > div > div:nth-child(1) > div > div.wkMJlb.YWi3ub > div > div.qZmL0 > div:nth-child(2) > c-wiz:nth-child(6) > section > header > div > div:nth-child(2) > button'
btn_selector = '#yDmH0d > c-wiz.SSPGKf.Czez9d > div > div > div:nth-child(1) > div > div.wkMJlb.YWi3ub > div > div.qZmL0 > div:nth-child(2) > c-wiz:nth-child(7) > section > header > div > div:nth-child(2) > button'
driver.find_element(By.CSS_SELECTOR, btn_selector).click()
time.sleep(1) # 팝업 로딩 대기

별5개: //*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[7]/div[2]/div[2]
별4개: //*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[6]/div[2]/div[2]
별3개: //*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[5]/div[2]/div[2]
별2개: //*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[4]/div[2]/div[2]
별1개: //*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[3]/div[2]/div[2]

In [36]:
# 3. 정렬 기준 변경: '평점'으로 교체
try:
    # (1) 정렬 드롭다운 메뉴 클릭
    
    #star_menu_xpath = '//*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div/div[1]/div/div/div/div[3]'
    star_menu_xpath = '//*[@id="yDmH0d"]/div[4]/div[2]/div/div/div/div/div[2]/div/div[1]/div/div/div/div[3]/div[2]/i'
    driver.find_element(By.XPATH, star_menu_xpath).click()
    time.sleep(1.5)

    # (2) 사용자님이 찾으신 '평점' 옵션 XPath로 클릭
    # 별점 5개: span[6]
    # 별점 1계: span[2]
    #rating_option_xpath = '//*[@id="yDmH0d"]/div[5]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[6]/div[2]/div[2]'
    rating_option_xpath = '//*[@id="yDmH0d"]/div[4]/div[2]/div/div/div/div/div[2]/div[2]/div/div/span[2]/div[2]/div[2]'
    driver.find_element(By.XPATH, rating_option_xpath).click()
    print("⭐ 정렬 기준을 '평점'으로 변경 완료!")
    time.sleep(3) # 데이터 갱신 대기

    # (3) 별 5개 필터가 풀렸을 수 있으니 다시 한 번 확인 (필요 시 필터 코드 재실행)
    # ... (별 5개 필터 클릭 코드) ...

except Exception as e:
    print(f"❌ 정렬 변경 실패: {e}")

⭐ 정렬 기준을 '평점'으로 변경 완료!


In [37]:
# 3. 크롤링 변수 및 스크롤 타겟 설정
target_count = 10000
collected_ids = set()
all_reviews = []

In [38]:
# [성공 포인트] 실제 스크롤이 일어나는 컨테이너 지정
popup_window = driver.find_element(By.CSS_SELECTOR, 'div.fysCi')

print(f"작업 시작: 목표 {target_count}개 수집")

# 4. 메인 크롤링 루프
while len(all_reviews) < target_count:
    # 현재 화면의 모든 리뷰 박스 로드
    review_boxes = driver.find_elements(By.CSS_SELECTOR, 'div.RHo1pe')
    
    new_found_this_turn = []
    
    # 역순 검사로 중복 제외한 새 리뷰만 필터링
    for i in range(len(review_boxes) - 1, -1, -1):
        try:
            header = review_boxes[i].find_element(By.CSS_SELECTOR, 'header.c1bOId')
            r_id = header.get_attribute('data-review-id')
            
            if r_id in collected_ids:
                break # 이미 수집한 구간에 도달하면 멈춤
            
            new_found_this_turn.append(review_boxes[i])
        except:
            continue

    # 새로운 리뷰 데이터 추출
    if new_found_this_turn:
        for box in reversed(new_found_this_turn):
            try:
                header = box.find_element(By.CSS_SELECTOR, 'header.c1bOId')
                r_id = header.get_attribute('data-review-id')
                user_name = box.find_element(By.CSS_SELECTOR, '.X5PpBb').text
                date = box.find_element(By.CSS_SELECTOR, '.bp9Aid').text
                rating = box.find_element(By.CSS_SELECTOR, '.iXRFPc').get_attribute('aria-label')
                content = box.find_element(By.CSS_SELECTOR, '.h3YV2d').text
                
                all_reviews.append({
                    'ID': r_id,
                    '작성자': user_name,
                    '날짜': date,
                    '별점': rating,
                    '내용': content
                })
                collected_ids.add(r_id)
            except:
                continue
        # ------------------------------------------------------
        # [여기가 삽입 위치!] 추출이 끝난 직후에 화면을 청소합니다.
        # ------------------------------------------------------
        driver.execute_script("""
            var reviews = document.querySelectorAll('div.RHo1pe');
            if (reviews.length > 50) {
                // 최신 50개만 남기고 이전 요소들은 브라우저 메모리에서 완전히 제거
                for (var i = 0; i < reviews.length - 50; i++) {
                    reviews[i].remove();
                }
            }
        """)
        print(f"                                     화면 최적화 완료 (메모리 정리) | 현재 총 수집: {len(all_reviews)} ", end='\r')
        # ------------------------------------------------------


        print(f"진행 현황: {len(all_reviews)} / {target_count} 완료", end='\r')

        # 500개마다 중간 저장
        if len(all_reviews) % 40 == 0:
            #pd.DataFrame(all_reviews).to_csv('roblox_reviews_progress.csv', index=False, encoding='utf-8-sig')
            pd.DataFrame(all_reviews).to_csv(
                'pikmin_reviews_progress.csv', 
                index=False, 
                encoding='utf-8-sig',
                escapechar='\\' )   # 특수문자 앞에 역슬래시를 붙여 에러를 방지합니다.
                                
    # 5. [동적 스크롤] 13번 PAGE_DOWN 수행
    for _ in range(12):
        popup_window.send_keys(Keys.PAGE_DOWN)
        # 매우 빠른 연타로 인한 씹힘 방지
        time.sleep(0.2)
    
    # 새 데이터 로딩을 위한 대기
    time.sleep(2.0)

    # 안전장치: 스크롤 후에도 개수가 늘어나지 않으면 종료
    #if len(driver.find_elements(By.CSS_SELECTOR, 'div.RHo1pe')) == len(review_boxes):
        # 네트워크 지연일 수 있으므로 마지막으로 5초 더 대기 후 확인
    #    time.sleep(5.0)
    #    if len(driver.find_elements(By.CSS_SELECTOR, 'div.RHo1pe')) == len(review_boxes):
    #        print("\n더 이상 새로운 리뷰가 없습니다. 수집을 종료합니다.")
    #        break

# 6. 최종 저장
df = pd.DataFrame(all_reviews)
df.to_csv('pikmin_reviews_final.csv', index=False, encoding='utf-8-sig')
print(f"\n작업 완료! 총 {len(df)}개의 리뷰가 저장되었습니다.")

driver.quit()

작업 시작: 목표 10000개 수집


KeyboardInterrupt: 